[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abuhuzaifahbidin/megat/blob/main/notebooks/03_pretokenize_and_upload.ipynb)

# Notebook 03 — Pre-tokenize Corpus & Upload to HuggingFace Hub

**What this notebook does:**
1. Loads the custom Malay tokenizer (from notebook 02)
2. Converts the full JSONL corpus → flat binary uint16 files (the format `train.py` reads)
3. Creates a train/val split
4. Uploads everything to a private HuggingFace Hub dataset repo
5. Shows you the RunPod download command

**Expected runtime:** 3–5 hours (tokenizing 4M docs is CPU-bound)

**Expected output:** `data/train_000.bin ... train_NNN.bin, val_000.bin` uploaded to HF Hub

---
**Before this notebook:** Run `02_train_tokenizer.ipynb`  
**After this notebook:** Push code to GitHub, spin up RunPod, and run `train.py`

---
### Why pre-tokenize to binary?

During training, the model processes ~512,000 tokens per step, and does ~15,000 steps.
That's ~7.68 billion token lookups. If we tokenized on-the-fly during training:
- Each document would be re-tokenized every time it's seen
- Tokenization is CPU-bound — it would bottleneck the GPU
- On an A40 doing 25,000 tokens/sec, the GPU would sit idle waiting for data

By pre-tokenizing once and saving as binary uint16:
- Training reads raw numbers directly — no string processing at all
- `np.memmap` loads only the pages being trained on (no RAM bottleneck)
- The GPU stays fed with data continuously

## Step 0 — Install packages

In [ ]:
!pip install -q tokenizers huggingface_hub

## Step 1 — Mount Drive, set paths, load tokenizer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Must match notebooks 01 and 02
SAVE_DIR      = '/content/drive/MyDrive/megat_data'
CORPUS_FILE   = os.path.join(SAVE_DIR, 'fineweb2_malay.jsonl')
WIKI_FILE     = os.path.join(SAVE_DIR, 'malay_wikipedia.jsonl')
TOKENIZER_DIR = os.path.join(SAVE_DIR, 'megat_tokenizer')
BIN_DIR       = os.path.join(SAVE_DIR, 'data')   # where .bin files go

os.makedirs(BIN_DIR, exist_ok=True)

for path in [CORPUS_FILE, TOKENIZER_DIR]:
    assert os.path.exists(path), f'Not found: {path} — run earlier notebooks first!'

print(f'Corpus:    {os.path.getsize(CORPUS_FILE)/1e9:.2f} GB')
print(f'Tokenizer: {os.listdir(TOKENIZER_DIR)}')
print(f'Output:    {BIN_DIR}')

In [ ]:
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer(
    os.path.join(TOKENIZER_DIR, 'vocab.json'),
    os.path.join(TOKENIZER_DIR, 'merges.txt'),
)

# The endoftext token is appended after every document during pre-tokenization.
# This teaches the model that documents are independent —
# it should not try to connect the ending of one document to the start of the next.
EOT_ID = tokenizer.token_to_id('<|endoftext|>')

print(f'Tokenizer loaded  |  vocab size: {tokenizer.get_vocab_size():,}')
print(f'EOT token ID: {EOT_ID}')

## Step 2 — Pre-tokenize corpus to binary files

We split the output into multiple files of ~50M tokens each (~100MB per file).
This is the format `MemmapDataLoader` in `train.py` expects:
```
data/
    train_000.bin  (~50M tokens, ~100MB)
    train_001.bin
    ...
    val_000.bin    (last chunk, set aside as validation)
```

Each `.bin` file is a flat array of `uint16` values (2 bytes per token).
`uint16` supports values 0–65,535 which comfortably covers our 32K vocab.

**Resume support:** If the session dies mid-way, the notebook detects already-saved
files and resumes from where it left off.

In [ ]:
import json
import time
import numpy as np

# ── Configuration ──────────────────────────────────────────────────────────────
TOKENS_PER_FILE = 50_000_000  # tokens per .bin file (50M × 2 bytes = 100MB)
VAL_FRACTION    = 0.005       # 0.5% of data goes to validation
                               # At 2B tokens that's 10M val tokens — plenty

# ── Check for already-saved files so we can resume ────────────────────────────
existing_files  = sorted([f for f in os.listdir(BIN_DIR) if f.endswith('.bin')])
tokens_already  = sum(
    os.path.getsize(os.path.join(BIN_DIR, f)) // 2
    for f in existing_files
)

if existing_files:
    print(f'Found {len(existing_files)} existing file(s), ~{tokens_already/1e9:.2f}B tokens already done.')
    print(f'Files: {existing_files}')
else:
    print('No existing files — starting fresh.')

In [ ]:
# ── Count total lines in corpus (to estimate total tokens and ETA) ─────────────
print('Counting corpus lines...')
total_lines = 0
with open(CORPUS_FILE, 'r') as f:
    for _ in f:
        total_lines += 1
print(f'Total documents: {total_lines:,}')

In [ ]:
# ── Main pre-tokenization loop ─────────────────────────────────────────────────

def get_all_docs(corpus_file, wiki_file=None):
    """Generator yielding text from corpus + optional Wikipedia."""
    with open(corpus_file, 'r') as f:
        for line in f:
            yield json.loads(line)['text']
    if wiki_file and os.path.exists(wiki_file):
        with open(wiki_file, 'r') as f:
            for line in f:
                yield json.loads(line)['text']


def save_chunk(chunk_tokens, file_idx, out_dir):
    """Save a list of token IDs as a uint16 binary file."""
    arr  = np.array(chunk_tokens, dtype=np.uint16)
    path = os.path.join(out_dir, f'train_{file_idx:03d}.bin')
    arr.tofile(path)
    size_mb = arr.nbytes / 1e6
    print(f'  Saved {path}  ({len(arr)/1e6:.1f}M tokens  {size_mb:.0f} MB)')
    return path


# We need to figure out where to resume.
# Each existing train_XXX.bin represents TOKENS_PER_FILE tokens already processed.
# Count how many complete chunks we already have.
existing_train = sorted([f for f in os.listdir(BIN_DIR) if f.startswith('train_') and f.endswith('.bin')])
file_idx       = len(existing_train)  # next file index to write
docs_to_skip   = file_idx * (TOKENS_PER_FILE // 550)  # approx docs per file (550 tokens/doc average)

print(f'Starting from file index {file_idx}, skipping ~{docs_to_skip:,} docs')

chunk_buffer  = []   # accumulate token IDs until we hit TOKENS_PER_FILE
total_tokens  = tokens_already
doc_count     = 0
t_start       = time.time()
t_last        = t_start

for doc_text in get_all_docs(CORPUS_FILE, WIKI_FILE):
    # Skip documents that were already processed in a previous session
    if doc_count < docs_to_skip:
        doc_count += 1
        continue

    # Tokenize the document
    token_ids = tokenizer.encode(doc_text).ids

    # Append the EOT token after every document.
    # This boundary is important: without it, the model would try to
    # predict across document boundaries, learning nonsense connections.
    token_ids.append(EOT_ID)

    chunk_buffer.extend(token_ids)
    total_tokens += len(token_ids)
    doc_count    += 1

    # When buffer is full, flush to disk
    if len(chunk_buffer) >= TOKENS_PER_FILE:
        save_chunk(chunk_buffer[:TOKENS_PER_FILE], file_idx, BIN_DIR)
        chunk_buffer = chunk_buffer[TOKENS_PER_FILE:]  # keep any overflow
        file_idx += 1

        # Progress report
        now    = time.time()
        rate   = (total_tokens - tokens_already) / (now - t_start) / 1000  # K tok/sec
        t_last = now
        print(f'  Progress: {doc_count:,} docs  {total_tokens/1e9:.2f}B tokens  {rate:.0f}K tok/s')

# Save the final (possibly partial) chunk
if chunk_buffer:
    save_chunk(chunk_buffer, file_idx, BIN_DIR)
    file_idx += 1

elapsed = (time.time() - t_start) / 60
print(f'\nPre-tokenization complete!')
print(f'  Total tokens: {total_tokens/1e9:.2f}B')
print(f'  Total files:  {file_idx}')
print(f'  Elapsed:      {elapsed:.0f} min')

## Step 3 — Create validation split

We designate the last training file as the validation set by renaming it.

Why the last file?
- Documents are roughly in crawl order (not shuffled), so the last file
  contains documents the model sees last during training — a fair val set.
- At ~50M tokens, the val set is ~2.5% of data — more than enough to get
  a statistically reliable loss estimate.

0.5% fraction means we'd normally want a smaller val file, but since our
files are chunked at 50M tokens, we just use one full file for validation.
That gives slightly more val data than the 0.5% target, which is fine.

In [ ]:
import glob

train_files = sorted(glob.glob(os.path.join(BIN_DIR, 'train_*.bin')))

if not train_files:
    print('ERROR: No train files found — pre-tokenization may not have run.')
elif os.path.exists(os.path.join(BIN_DIR, 'val_000.bin')):
    print('val_000.bin already exists — skipping split.')
else:
    # Move the last training file to val_000.bin
    last_train = train_files[-1]
    val_path   = os.path.join(BIN_DIR, 'val_000.bin')
    os.rename(last_train, val_path)

    val_tokens = os.path.getsize(val_path) // 2
    print(f'Validation split: {last_train} → val_000.bin')
    print(f'Val tokens: {val_tokens/1e6:.1f}M')

# Final count
train_bins = sorted(glob.glob(os.path.join(BIN_DIR, 'train_*.bin')))
val_bins   = sorted(glob.glob(os.path.join(BIN_DIR, 'val_*.bin')))

train_tokens = sum(os.path.getsize(f) // 2 for f in train_bins)
val_tokens   = sum(os.path.getsize(f) // 2 for f in val_bins)

print(f'\n── Data split summary ─────────────────────────────────')
print(f'  Train files: {len(train_bins)}  ({train_tokens/1e9:.2f}B tokens)')
print(f'  Val files:   {len(val_bins)}   ({val_tokens/1e6:.1f}M tokens)')
print(f'  Total:       {(train_tokens + val_tokens)/1e9:.2f}B tokens')

## Step 4 — Upload to HuggingFace Hub

We upload the binary data files and tokenizer to a private HuggingFace dataset repo.
From RunPod, one command pulls everything down in minutes over their fast network.

**Before running this cell:**
1. Create a free account at huggingface.co
2. Go to Settings → Access Tokens → New token (write permission)
3. Paste it when prompted below
4. Replace `YOUR_HF_USERNAME` with your actual HF username

In [ ]:
from huggingface_hub import HfApi, login

# Log in with your HuggingFace token.
# You'll be prompted to paste it — it won't appear in the notebook output.
login()

api = HfApi()

In [ ]:
# ── Configuration — CHANGE THIS ────────────────────────────────────────────────
HF_USERNAME = 'YOUR_HF_USERNAME'  # ← replace with your HuggingFace username
REPO_NAME   = 'megat-pretrain-data'
REPO_ID     = f'{HF_USERNAME}/{REPO_NAME}'

print(f'Repo: {REPO_ID}')

In [ ]:
# Create the dataset repository (private so your data stays yours)
# If it already exists, this is a no-op.

try:
    api.create_repo(REPO_ID, repo_type='dataset', private=True)
    print(f'Created repo: {REPO_ID}')
except Exception as e:
    print(f'Repo already exists or error: {e}')

In [ ]:
# Upload all binary data files.
# This may take 30–60 minutes depending on Colab's upload speed.
# Each file is ~100MB. If it fails mid-way, re-run — HF Hub handles duplicates.

all_bins = sorted(glob.glob(os.path.join(BIN_DIR, '*.bin')))
print(f'Uploading {len(all_bins)} binary files...')

for i, bin_path in enumerate(all_bins):
    filename = os.path.basename(bin_path)
    size_mb  = os.path.getsize(bin_path) / 1e6
    print(f'  [{i+1}/{len(all_bins)}] {filename} ({size_mb:.0f} MB) ...', end=' ')

    api.upload_file(
        path_or_fileobj=bin_path,
        path_in_repo=f'data/{filename}',
        repo_id=REPO_ID,
        repo_type='dataset',
    )
    print('done')

print()
print('All data files uploaded.')

In [ ]:
# Upload tokenizer files
tokenizer_files = os.listdir(TOKENIZER_DIR)
print(f'Uploading tokenizer: {tokenizer_files}')

for filename in tokenizer_files:
    filepath = os.path.join(TOKENIZER_DIR, filename)
    api.upload_file(
        path_or_fileobj=filepath,
        path_in_repo=f'megat_tokenizer/{filename}',
        repo_id=REPO_ID,
        repo_type='dataset',
    )
    print(f'  Uploaded: megat_tokenizer/{filename}')

print('\nTokenizer uploaded.')

## Step 5 — Verify the upload

In [ ]:
# List files in the Hub repo to confirm everything is there
repo_files = api.list_repo_files(REPO_ID, repo_type='dataset')

data_files      = [f for f in repo_files if f.startswith('data/')]
tokenizer_files = [f for f in repo_files if f.startswith('megat_tokenizer/')]

print(f'── HuggingFace Hub: {REPO_ID} ────────────────────')
print(f'  Data files:      {len(data_files)}')
for f in sorted(data_files):
    print(f'    {f}')
print(f'  Tokenizer files: {len(tokenizer_files)}')
for f in sorted(tokenizer_files):
    print(f'    {f}')

has_val       = any('val_000' in f for f in data_files)
has_train     = any('train_' in f for f in data_files)
has_vocab     = any('vocab.json' in f for f in tokenizer_files)
has_merges    = any('merges.txt' in f for f in tokenizer_files)

print()
print(f'  val_000.bin:  {"✓" if has_val else "✗ MISSING"}')
print(f'  train files:  {"✓" if has_train else "✗ MISSING"}')
print(f'  vocab.json:   {"✓" if has_vocab else "✗ MISSING"}')
print(f'  merges.txt:   {"✓" if has_merges else "✗ MISSING"}')

## Step 6 — RunPod setup commands

Print the exact commands you'll run after spinning up the RunPod A40.

In [ ]:
print('═' * 60)
print('  Phase 0 complete! Data is on HuggingFace Hub.')
print('═' * 60)
print()
print('When your RunPod A40 instance starts, run these commands:')
print()
print('  # 1. Install dependencies')
print('  pip install tokenizers huggingface_hub wandb')
print()
print('  # 2. Log in to HuggingFace')
print('  huggingface-cli login')
print()
print('  # 3. Pull the repo with your training code')
print('  git clone https://github.com/abuhuzaifahbidin/megat.git')
print('  cd megat')
print()
print('  # 4. Download the pre-tokenized data and tokenizer')
print(f'  huggingface-cli download {REPO_ID} --repo-type dataset --local-dir ./')
print()
print('  # 5. Start training inside tmux (protects against SSH disconnects)')
print('  tmux new -s train')
print('  python train.py')
print()
print('  # 6. Detach from tmux (training continues in background)')
print('  Ctrl+B, then D')
print()
print('  # To reattach later:')
print('  tmux attach -t train')
print()
print('═' * 60)
print('  Expected loss at start: ~10.4 (log(32000) = random weights)')
print('  Expected loss at hour 12: ~3.5–4.0')
print('  Expected loss at hour 36: ~3.0–3.5')
print('  Expected loss at hour 72: ~2.8–3.2')
print('═' * 60)